In [4]:
import pandas as pd

rides=pd.read_parquet("data/transformed/validated_rides_2022_01.parquet")

rides.head(10)

,pickup_datetime,pickup_location_id
0,2022-01-01 00:35:40,142
1,2022-01-01 00:33:43,236
2,2022-01-01 00:53:21,166
3,2022-01-01 00:25:21,114
4,2022-01-01 00:36:48,68
5,2022-01-01 00:40:15,138
6,2022-01-01 00:20:50,233
7,2022-01-01 00:13:04,238
8,2022-01-01 00:30:02,166
9,2022-01-01 00:48:52,236


In [5]:
# rounding times to closest previous hour, cause we need the data on an hourly frequency

rides['pickup_hour'] = rides['pickup_datetime'].dt.floor('h')
rides

,pickup_datetime,pickup_location_id,pickup_hour
0,2022-01-01 00:35:40,142,2022-01-01 00:00:00
1,2022-01-01 00:33:43,236,2022-01-01 00:00:00
2,2022-01-01 00:53:21,166,2022-01-01 00:00:00
3,2022-01-01 00:25:21,114,2022-01-01 00:00:00
4,2022-01-01 00:36:48,68,2022-01-01 00:00:00
...,...,...,...
2463926,2022-01-31 23:36:53,90,2022-01-31 23:00:00
2463927,2022-01-31 23:44:22,107,2022-01-31 23:00:00
2463928,2022-01-31 23:39:00,113,2022-01-31 23:00:00
2463929,2022-01-31 23:36:42,148,2022-01-31 23:00:00


In [6]:
# grouping rides per location per hour to know how many rides were there in a specific location on a specific time
agg_rides=rides.groupby(['pickup_hour','pickup_location_id']).size().reset_index()
agg_rides.rename(columns={0:'rides'}, inplace=True)
agg_rides

,pickup_hour,pickup_location_id,rides
0,2022-01-01 00:00:00,4,11
1,2022-01-01 00:00:00,7,6
2,2022-01-01 00:00:00,10,1
3,2022-01-01 00:00:00,12,2
4,2022-01-01 00:00:00,13,12
...,...,...,...
66863,2022-01-31 23:00:00,261,4
66864,2022-01-31 23:00:00,262,8
66865,2022-01-31 23:00:00,263,26
66866,2022-01-31 23:00:00,264,24


In [7]:
# adding the rows with zero rides to get the complete time-series data.
from tqdm import tqdm
def add_missing_slots(agg_rides: pd.DataFrame) -> pd.DataFrame:
    location_ids=agg_rides['pickup_location_id'].unique()
    full_range=pd.date_range(agg_rides['pickup_hour'].min(), agg_rides['pickup_hour'].max(), freq='h')
    output=pd.DataFrame()

    for location_id in tqdm(location_ids):

        #keep only rides for this location id
        agg_rides_i= agg_rides.loc[agg_rides.pickup_location_id == location_id, ['pickup_hour', 'rides']]

        # quick way to add missing data with 0 in a Series
        agg_rides_i.set_index('pickup_hour', inplace=True)
        agg_rides_i.index=pd.DatetimeIndex(agg_rides_i.index)
        agg_rides_i=agg_rides_i.reindex(full_range, fill_value=0)

        agg_rides_i['pickup_location_id']=location_id

        # add back 'location_id' columns
        output=pd.concat([output, agg_rides_i])
    output=output.reset_index().rename(columns={'index':'pickup_hour'})
    return output

In [8]:
agg_rides_all_slots= add_missing_slots(agg_rides)

100%|██████████| 257/257 [00:01<00:00, 178.68it/s]


In [9]:
# visualising the data to detect bugs in our data processing.
from typing import Optional, List
import plotly.express as px
def plot_rides (
        rides: pd.DataFrame(),
        locations: Optional[List[int]] = None
):
    """
    Plot time-series data
    """
    rides_to_plot = rides[rides.pickup_location_id.isin(locations)] if locations else rides

    fig = px.line(
        rides_to_plot,
        x="pickup_hour",
        y="rides",
        color="pickup_location_id",
        template="none",
    )  

    fig.show()

In [16]:
# plot the data for cental park
plot_rides(agg_rides_all_slots, locations=[43])

In [10]:
# saving our time-series data into disk
agg_rides_all_slots.to_parquet("data/transformed/ts_data_2022_01.parquet")